In [ ]:
# Library imports
import polars as pl
import os   
import dill as pickle
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
# Load position info
video_df = pl.read_csv(r"E:\efizz\JAL006\JAL006_shelter_barrier_flip_6_2024_03_28T10_54_20\processed_data\full_video_dataframe.csv")
homie_path = r"E:\efizz\JAL006\JAL006_shelter_barrier_flip_6_2024_03_28T10_54_20\processed_data\homings\homings_obj.pkl"
with open(homie_path, "rb") as dill_file:
    homings = pickle.load(dill_file)
frame_by_cluster_matrix = np.load(r"E:\efizz\JAL006\JAL006_shelter_barrier_flip_6_2024_03_28T10_54_20\processed_data\frame_by_synthetic_cluster_matrix.npy")
video_and_spike_data = pl.read_parquet(r"E:\efizz\JAL006\JAL006_shelter_barrier_flip_6_2024_03_28T10_54_20\processed_data\synthetic_video_spike_count_df")

In [ ]:
print(frame_by_cluster_matrix)
print(frame_by_cluster_matrix.shape)
print(video_df.shape)
print(video_df.columns)

In [ ]:
# Because of memory constraints, let's only keep the columns we need for this module
COLUMNS_TO_KEEP = [
    "frames",
    "mouse_x_position",
    "mouse_y_position",
    "OutofshelterIdx",
    "EscapePeriod",
    "shelter",
    "hdir",
    "barrier_present",
    "barrier_flipped",
    'hsa', 
    'h_bar_north_a', 
    'h_bar_south_a',
]

# Select some columns to keep
 

In [ ]:
video_df = video_df.select(COLUMNS_TO_KEEP)  # Prevents memory issues and computer crashes
# video_df = video_df.filter((video_df["barrier_present"] == True) & (video_df["barrier_flipped"] == False))
print(video_df)
print(video_df.columns)

In [ ]:
homings

In [ ]:
onset_frames = homings.onset_frames

In [ ]:
plt.scatter(video_df["mouse_x_position"], video_df["mouse_y_position"])

## A function to check the attributes required from the homing object exsists and to put them into a neat dictionary

In [ ]:
def access_homing_attributes(homings_obj: object) -> dict:
    """Return a dictionary of verified homing attributes"""
    try:
        onset_frames = homings_obj.onset_frames
        offset_frames = homings_obj.offset_frames
    except AttributeError:
        raise AttributeError("The homings object does not have the required attributes - Something upstream is wrong with the homings object")
    assert len(onset_frames) == len(offset_frames), "The onset and offset frames are not the same length - Something is wrong with the homings object"
    return {"onset_frames": onset_frames, "offset_frames": offset_frames}

In [ ]:
dic = access_homing_attributes(homings)
print(dic)
print(len(dic["onset_frames"]))

In [ ]:
def plot_homing_durations(homings_dic) -> None:
    """Plot the durations of homings"""
    onset_frames = homings_dic["onset_frames"]
    offset_frames = homings_dic["offset_frames"]
    durations = (offset_frames - onset_frames) / 40 # 40 fps
    plt.hist(durations)
    plt.xlabel("Duration (s)")
    plt.ylabel("Number of homings")
    plt.title("Homing durations")
    plt.show()
plot_homing_durations(dic)

In [ ]:
def check_frames_increment_by_one(arr):
    """A test to check that frames increment by 1 and as such is continuous ensuring no frames are skipped"""
    # Check each element to see if it increments by 1
    for i in range(len(arr) - 1):
        if arr[i + 1] - arr[i] != 1:
            return False
    return True

### Create some distance bins to see distribution across y axis

In [ ]:
ycoords = video_df["mouse_y_position"]
# plt.hist(ycoords, bins=16)
# now remove y coordinats greater than 800
ycoords = ycoords.filter(ycoords < 800)
# plt.hist(ycoords, bins=16)
plt.title("Distribution of y coordinates in bins")
# 8 bins would be 10cm each, so 16 bins would be 5cm each
bins = np.linspace(0, 800, 32) # Remove near shelter as there are a lot of frames there
plt.hist(ycoords, bins=bins, color="green") # These are the defined bins
print(bins)
# NOTE there seems to be more frames around the subgoal probably due to the turn, so we should
# probably sample the frames more uniformly across bins depending on how we want to analyse the data
# TODO - Speak to jas about how she uniformly samples across bins

In [ ]:
def extract_homing_info(dic, video_df):
    """Extract homing info from video_df"""
    onset_frames = dic["onset_frames"]
    offset_frames = dic["offset_frames"]
    homing_info = []
    for onset, offset in zip(onset_frames, offset_frames):
        homing = video_df[onset:offset]
        homing = homing.select(["frames", "mouse_x_position", "mouse_y_position", "hdir",  'hsa', 'h_bar_north_a', 'h_bar_south_a',])
        homing_info.append(homing)
        
    # Check the frame column of each homing information increments uniformly by 1 such that no frames are missed
    for homing in homing_info:
        assert check_frames_increment_by_one(homing["frames"].to_numpy()), "Frames are missing in the homing information"
    
    return homing_info
info = extract_homing_info(dic, video_df)
print("Printing an example of the homing info")
print(info[3])

In [ ]:
def select_similar_homings(info):
    """
    
    -- Similar time periods
    -- Similar mouse positions
    -- Similar targets

    Raises:
        NotImplementedError: _description_
    """
    # HARDCORE MODE - we like it rough and tough
    xcoordinate_min = 300
    xcoordinate_max = 700
    ycoordinate_min = 200
    ycoordinate_end_min = 750
    x_middle_chunk_min = 400
    x_middle_chunk_max = 600
    extracted_info = []
    for idx, homing in enumerate(info):
        
        # check if mouse x position is within the range - STARTS FARTS ONLY 
        start_x = homing["mouse_x_position"][0]
        start_y = homing["mouse_y_position"][0]
        
        # Starts in a similar space
        if start_x > xcoordinate_min and start_x < xcoordinate_max and start_y < ycoordinate_min:
            
            #Ends in a similar space
            if homing["mouse_y_position"][-1] > ycoordinate_end_min:
                
                # middle of frames is in a similar space
                middle_x = homing["mouse_x_position"][int(len(homing) / 2)]
                if middle_x < x_middle_chunk_min or middle_x > x_middle_chunk_max:
                    # plt.scatter(homing["mouse_x_position"], homing["mouse_y_position"])
                    # plt.hlines(y=512, xmin=150, xmax=900, color="k")
                    
                    # CHOOSE ONE SIDE
                    if middle_x > 700:
                        plt.scatter(homing["mouse_x_position"], homing["mouse_y_position"])
                        plt.hlines(y=512, xmin=150, xmax=900, color="k")
                        plt.vlines(x=700, ymin=50, ymax=900, color="k")
                    
                    
                        extracted_info.append(homing)
                    
    return extracted_info

extracted_info =  select_similar_homings(info)
print(extracted_info[0])
print(len(extracted_info))

# Plot the hdir arrwos at each point

In [ ]:
length = 20
dx = length * np.cos(extracted_info[0]["hdir"])
dy = length * -np.sin(extracted_info[0]["hdir"])
plt.figure(figsize=(10, 6))
xs = extracted_info[0]["mouse_x_position"]
ys = extracted_info[0]["mouse_y_position"]
plt.quiver(xs, ys, dx, dy, angles='xy', scale_units='xy', scale=2, color='blue')
plt.xlim(xs.min() - 1, xs.max() + 1)
plt.ylim(ys.min() - 1, ys.max() + 1)
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.title('Vector Field of Arrows')
plt.grid(True)
plt.show()


### Angle operations

In [ ]:
def circular_difference(angle1: np.ndarray, angle2: np.ndarray) -> np.ndarray:
    """Calculates the shortest difference between two angles in radians from the origin"""

    # If the angle is greater than pi then subtract 2pi to get the smallest difference from the origin
    angle1 = np.where(angle1 > np.pi, (2 * np.pi) - angle1, angle1)
    angle2 = np.where(angle2 > np.pi, (2 * np.pi) - angle2, angle2)
    diff = np.arctan2(np.sin(angle1 - angle2), np.cos(angle1 - angle2))

    return diff
    
def circular_sum(angle1, angle2):
    """Takes in radian values between 0 and 2pi as scalars or vectors and returns the circular sum."""
    # if the angle is greater than pi then subtract 2pi to get the smallest difference
    angle1 = np.where(angle1 > np.pi, (2 * np.pi) - angle1, angle1)
    angle2 = np.where(angle2 > np.pi, (2 * np.pi) - angle2, angle2)
    
    sum_angle = np.arctan2(np.sin(angle1 + angle2), np.cos(angle1 + angle2))
    # Adjust the result to be between 0 and 2pi because the arctan2 function returns values between -pi and pi
    asjusted_result = np.where(sum_angle < 0, sum_angle + 2 * np.pi, sum_angle)
    return asjusted_result
    
def compute_predictor(angle1: np.ndarray, angle2: np.ndarray) -> np.ndarray:
    """Compute a normalised predictor between -1 and 1 between two angles:

    Args:
        angle1: The first angle in radians
        angle2: The second angle in radians

    Metric is computed as:
       -1: The angle to angle1 is close to zero and the angle2 is close to pi
        0: The angle to angle1 is close to the angle to angle2
        1: The angle to angle1 is close to pi and the angle to angle2 is close to zero
    """
    numerator = circular_difference(angle1, angle2)
    denominator = circular_sum(angle1, angle2)
    return numerator / denominator

### Tests tests test PIGS PIGS PIGS 


In [ ]:
# -------------------------------- UNIT TESTS SIMPLE -------------------------------------------------------------
test_hsa = np.array([0, np.pi, np.pi, np.pi/12, (5*np.pi)/6, (11*np.pi)/12, np.pi/12, (23*np.pi)/12])
test_angle = np.array([np.pi, 0, np.pi, (5*np.pi)/6, np.pi/12, (11*np.pi)/12, (13*np.pi)/12, (7*np.pi)/6])
test_result = compute_predictor(test_hsa, test_angle)

assert test_result[0] == -1, "Test 1 failed, if mouse face shelter then expected -1 but got {}".format(test_result[0])
assert test_result[1] == 1, "Test 2 failed, if mouse face the test goal expected 1 but got {}".format(test_result[1])
assert test_result[2] == 0, "Test 3 failed, expected 0 but got {}".format(test_result[2])
assert np.around(test_result[3], 1) == -0.8, "Test 4 failed, expected -0.8 but got {}. Should be negative as mouse facing closer to shelter".format(test_result[3]) 
assert np.around(test_result[4], 1) == 0.8, "Test 4 failed, expected 0.8 but got {}. Should be positive as mouse facing closer to goal".format(test_result[3]) 
assert test_result[5] == 0, "Test 5 failed, expected 0 as angles are the same but got {}".format(test_result[5])
assert np.around(test_result[6], 1) == -0.8, "Test 6 failed, expected -0.8 but got {}. Answer should be closer to -0.9 as mouse is facing towards shelter ".format(test_result[6])
assert np.around(test_result[7], 1) == -0.8, "Test 7 failed, expected -0.8 but got {}. Answer should be closer to -0.8 as mouse is facing towards shelter ".format(test_result[7])
print("All tests passed")

## Compute the predictor


In [ ]:
import numpy as np
import polars as pl

predictors = []

# ------------------------------------------------------------------------------------------------------------

homing_data = {}
for idx, homing in enumerate(extracted_info):
    hsa = homing["hsa"].to_numpy().copy()
    goal = homing["h_bar_south_a"].to_numpy().copy()
    
    # Quality check
    # Check arrays are not above or below -pi and pi
    assert np.all(hsa >= -np.pi) and np.all(hsa <= np.pi), "hsa values are not within the range of -pi and pi"
    assert np.all(goal >= -np.pi) and np.all(goal <= np.pi), "h_bar_north_a values are not within the range of -pi and pi"
    
    # if values negative radians then add 2pi to make them positive
    # turn negative radians into positive radians to make them easier to work with
    hsa = np.where(hsa < 0, hsa + 2 * np.pi, hsa)
    goal = np.where(goal < 0, goal + 2 * np.pi, goal)
        
    predictor = compute_predictor(hsa, goal)
    assert np.all(predictor >= -1) and np.all(predictor <= 1), "Predictor values are not within the range of -1 and 1"
    predictors.append(predictor)
    homing = homing.with_column(pl.Series("predictor", predictor))
    homing_data[idx] = homing

# Example output
# # remove row limits to see all the data
pl.Config.set_tbl_rows(1000)

# flatten predictors
predictors = np.concatenate(predictors)

print(homing_data[0])  # Display the data with predictors for the first dataset
plt.hist(predictors)
plt.title("Predictor distribution")
plt.xlabel("Predictor value")
plt.ylabel("Frame count")
plt.show()


# Plot predictor for first homing

In [ ]:
test_homing = 3
length = 20
dx = length * np.cos(homing_data[test_homing]["hdir"])
dy = length * -np.sin(homing_data[test_homing]["hdir"])
plt.figure(figsize=(10, 6))
xs = homing_data[test_homing]["mouse_x_position"]
ys = homing_data[test_homing]["mouse_y_position"]
plt.quiver(xs, ys, dx, dy, angles='xy', scale_units='xy', scale=2, color='blue')

df = homing_data[test_homing].to_pandas()
print(homing_data[test_homing])

for i, row in df.iterrows():
    plt.text(x = row['mouse_x_position'] + 100, 
             y = row['mouse_y_position'] + 10, 
             s = str(np.around(row['predictor'], 1)), 
             color='red',
             fontsize=6)
    
#     # Print the goal angle
#     plt.text(x = row['mouse_x_position'] + 150, 
#             y = row['mouse_y_position'] + 10, 
#             s = str(np.around(row['h_bar_south_a'], 1)), 
#             color='black',
#             fontsize=6)
    
#     # Print the hsa angle
#     plt.text(x = row['mouse_x_position'] + 200, 
#             y = row['mouse_y_position'] + 10, 
#             s = str(np.around(row['hsa'], 1)), 
#             color='black',
#             fontsize=6)

plt.xlim(xs.min() - 1, xs.max() + 1)
plt.ylim(ys.min() - 1, ys.max() + 1)
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.title('Vector Field of Arrows')
plt.grid(True)
plt.show()

# Extract predictor across all homings and get predictor stats

In [ ]:
# Loop through the homing data and create a new array of all the predictors together
extracted_predictor = []
for idx, homing in enumerate(homing_data.values()):
    predictor = homing["predictor"]
    extracted_predictor.append(predictor.to_numpy())
flatten_extracted_predictor = np.concatenate(extracted_predictor)

# Pull at distribution statistics
median = np.nanmedian(flatten_extracted_predictor)
iqr = np.nanpercentile(flatten_extracted_predictor, 75) - np.nanpercentile(flatten_extracted_predictor, 25)
mean = np.nanmean(flatten_extracted_predictor)
std = np.nanstd(flatten_extracted_predictor)
print("The mean, median, std and IQR of the predictor are: ", mean, median, std, iqr)

# Plot the distribution of the predictors
plt.hist(flatten_extracted_predictor)
plt.yscale('log')
plt.title("Distribution of predictors on log scale")

### Just select the columns required E.G the frames and the predictor for each homing

In [ ]:
# get frames first
homing_dic_thinned = {}

for idx, homing in enumerate(homing_data.values()):
    assert len(homing) > 0, "The homing data is empty, something is wrong with the data"
    homing_dic_thinned[idx] = homing.select(["frames", 'predictor', "hsa", "hdir", "h_bar_north_a", "h_bar_south_a"])
print(homing_dic_thinned[0][:6]) # just show top few rows
    

## Create design matrix

In [ ]:
x = np.arange(10)
np.random.shuffle(x)
print(x)

# Split up the homing data such that we can leave one out for test

In [ ]:
def create_test_train_split_by_homings(data, number_of_test_homings):
    number_of_homings = len(data)
    homing_indices = list(data.keys())
    print(f"Homing_indices {homing_indices}")
    np.random.shuffle(homing_indices)
    print(f"Shuffled homing indicies", homing_indices)
    test = {}
    train = {}
    
    for idx, homing in enumerate(data.values()):
        homing = data[homing_indices[idx]]
        if idx < number_of_test_homings:
            # Assign the first few homings to the test set
            test[idx] = homing
        else:
            # Assign the rest to the train set
            train[idx] = homing

    return train, test
train, test = create_test_train_split_by_homings(homing_dic_thinned, number_of_test_homings = 1)
# print(train)
assert len(train) + len(test) == len(homing_dic_thinned), "The number of homings in the train and test set do not add up to the total number of homings"

# train and test are a dictionary of dataframes where the key is the index of the homing
# check that the dataframe in the train and test set are not the same
# logic only works if nubmer of test homings is 1
for idx in train.keys():
    assert train[idx] is not test[0], "The train and test dataframes are the same"

print(f"The number of hold out homings is: {len(test)}")

# A function to create the design matrix and depedent var vector

In [ ]:
def create_design_matrix_and_dependent_vector(data):
    
    # initialise the design matrix and dependent variable
    # print(data.values())
    total_frames = sum([len(homing["frames"]) for homing in data.values()])
    total_neurons = frame_by_cluster_matrix.shape[1] # Total number of neurons
    design_matrix = np.zeros((total_frames, total_neurons))
    dependent_var = np.zeros((total_frames)).T
    assert design_matrix.shape[0] == dependent_var.shape[0], "The design matrix and depedent vector are not the same length"
    
    # print(data)
    counter = 0
    for homing in data.values():
        
        # Extract the frames and predictors
        frames = homing["frames"].to_numpy()
        predictor = homing["predictor"].to_numpy()
        
        # Assign
        design_matrix[0+counter:counter+len(frames)] = frame_by_cluster_matrix[frames[0]:frames[-1]+1]
        dependent_var[0+counter:counter+len(frames)] = predictor.T
        counter += len(frames)
    
    return design_matrix, dependent_var

test_design_matrix, test_dependent_var = create_design_matrix_and_dependent_vector(test)
train_design_matrix, train_dependent_var = create_design_matrix_and_dependent_vector(train)
# Print the shapes
print(test_design_matrix.shape, test_dependent_var.shape)
print(train_design_matrix.shape, train_dependent_var.shape)

In [ ]:
from scipy.stats import zscore

# Initialise the design matrix and predictor vector
total_frames = sum([len(homing) for homing in homing_dic_thinned.values()]) # Total number of frames across all homings
total_neurons = frame_by_cluster_matrix.shape[1] # Total number of neurons
print(f"Shape of the design matrix is: {total_frames, total_neurons}")
design_matrix = np.zeros((total_frames, total_neurons))
predictor_vector = np.zeros((total_frames))
print("The shape of the predictor vector is: ", predictor_vector.shape)
assert design_matrix.shape[0] == predictor_vector.shape[0], "The design matrix and predictor vector are not the same length"
# ---------------------------------------------------------------------------------------------------------------------------

number_of_homings = len(homing_dic_thinned)

counter = 0
for idx, homing in enumerate(homing_dic_thinned.values()):
    # Extract
    frames = homing["frames"].to_numpy()
    predictor = homing["predictor"].to_numpy()
    
    # predictor = homing["normalised_predictor"].to_numpy()
    # predictor = homing["hsa"].to_numpy() # This is just for testing purposes
    # predictor = homing["hdir"].to_numpy() # This is just for testing purposes
    # predictor = homing["h_bar_north_a"].to_numpy() # This is just for testing purposes
    # predictor = homing["h_bar_south_a"].to_numpy() # This is just for testing purposes
    
    # Assign
    design_matrix[0+counter:counter+len(frames)] = frame_by_cluster_matrix[frames[0]:frames[-1]+1]
    predictor_vector[0+counter:counter+len(frames)] = predictor
    counter += len(frames)

# Check matrix rank
rank = np.linalg.matrix_rank(design_matrix)
if rank < design_matrix.shape[1]:
    print("The design matrix is rank deficient")

print(f"The rank of the design matrix is: {rank}")
print("The design matrix and predictor vector are ready")

#  ---------------------------------------------------------------------------- Sythentic tests ----------------------------------------------------------


# Generate and fit some easy sythetic data


In [ ]:
# Given a synthetically generated design matrix and dependent variable, we can now fit a linear model to the data.
def generate_synthetic_data(total_frames, total_neurons, influential_neurons):
    """Create a synthetic design matrix and dependent vector based on 
    
    
    Generates a synthetic design matrix and dependent vector.
    
    Parameters:
    - total_frames: int, number of frames or samples in the dataset.
    - total_neurons: int, number of neurons.
    - influential_neurons: int, number of neurons that directly influence the dependent variable.
    
    Returns:
    - design_matrix: numpy.ndarray, shape [total_frames, total_neurons]
    - dependent_var: numpy.ndarray, shape [total_frames]
    """
    assert influential_neurons <= total_neurons, "The number of influential neurons cannot be greater than the total number of neurons"
    
    np.random.seed(42)  # for reproducibility
    
    # Simulate neural activity: Each column in the design matrix corresponds to a neuron's spike counts
    design_matrix = np.random.poisson(lam=5, size=(total_frames, total_neurons))
    
    # Create coefficients for a subset of neurons to influence the dependent variable
    coefficients = np.zeros(total_neurons)
    influential_indices = np.random.choice(range(total_neurons), size=influential_neurons, replace=False) # replace=False ensures no neuron is selected twice
    coefficients[influential_indices] = np.random.uniform(-1, 1, size=influential_neurons)
    
    # Generate the dependent variable as a noisy linear combination of influential neurons' activities
    dependent_var = design_matrix @ coefficients + np.random.normal(0, 0.5, total_frames)  # adding some noise
    dependent_var = 2 * (dependent_var - np.min(dependent_var)) / (np.max(dependent_var) - np.min(dependent_var)) - 1  # scale to [-1, 1]
    
    return design_matrix, dependent_var

# Tests
design_matrix_robo_TN100_IN_1, dependent_var_robo_TN100_IN_1 = generate_synthetic_data(total_frames=1000, total_neurons=100, influential_neurons=1)
design_matrix_robo_TN500_IN_1, dependent_var_robo_TN500_IN_1 = generate_synthetic_data(total_frames=1000, total_neurons=500, influential_neurons=1)
design_matrix_robo_TN10_IN_5, dependent_var_robo_TN10_IN_5 = generate_synthetic_data(total_frames=1000, total_neurons=10, influential_neurons=5)
design_matrix_robo_TN10_IN_9, dependent_var_robo_TN10_IN_9 = generate_synthetic_data(total_frames=1000, total_neurons=10, influential_neurons=9)
design_matrix_robo_TN500_IN_250, dependent_var_robo_TN500_IN_250 = generate_synthetic_data(total_frames=1000, total_neurons=500, influential_neurons=250)
design_matrix_robo_TN500_IN_50, dependent_var_robo_TN500_IN_50 = generate_synthetic_data(total_frames=1000, total_neurons=500, influential_neurons=50)

# Fit a linear regression model per combination to check the affect of the number of neurons and influential neurons
reg_TN100_IN_1 = LinearRegression().fit(design_matrix_robo_TN100_IN_1, dependent_var_robo_TN100_IN_1)
r2_TN100_IN_1 = reg_TN100_IN_1.score(design_matrix_robo_TN100_IN_1, dependent_var_robo_TN100_IN_1)
print("The R2 score with 100 neurons and 1 influential neuron is: ", np.around(r2_TN100_IN_1, 2))

reg_TN500_IN_1 = LinearRegression().fit(design_matrix_robo_TN500_IN_1, dependent_var_robo_TN500_IN_1)
r2_TN500_IN_1 = reg_TN500_IN_1.score(design_matrix_robo_TN500_IN_1, dependent_var_robo_TN500_IN_1)
print("The R2 score with 500 neurons and 1 influential neuron is: ", np.around(r2_TN500_IN_1, 2))

reg_TN10_IN_5 = LinearRegression().fit(design_matrix_robo_TN10_IN_5, dependent_var_robo_TN10_IN_5)
r2_TN10_IN_5 = reg_TN10_IN_5.score(design_matrix_robo_TN10_IN_5, dependent_var_robo_TN10_IN_5)
print("The R2 score with 10 neurons and 5 influential neurons is: ", np.around(r2_TN10_IN_5, 2))

reg_TN10_IN_9 = LinearRegression().fit(design_matrix_robo_TN10_IN_9, dependent_var_robo_TN10_IN_9)
r2_TN10_IN_9 = reg_TN10_IN_9.score(design_matrix_robo_TN10_IN_9, dependent_var_robo_TN10_IN_9)
print("The R2 score with 10 neurons and 9 influential neurons is: ", np.around(r2_TN10_IN_9, 2))

reg_TN500_IN_250 = LinearRegression().fit(design_matrix_robo_TN500_IN_250, dependent_var_robo_TN500_IN_250)
r2_TN500_IN_250 = reg_TN500_IN_250.score(design_matrix_robo_TN500_IN_250, dependent_var_robo_TN500_IN_250)
print("The R2 score with 500 neurons and 250 influential neurons is: ", np.around(r2_TN500_IN_250, 2))

reg_TN500_IN_50 = LinearRegression().fit(design_matrix_robo_TN500_IN_50, dependent_var_robo_TN500_IN_50)
r2_TN500_IN_50 = reg_TN500_IN_50.score(design_matrix_robo_TN500_IN_50, dependent_var_robo_TN500_IN_50)
print("The R2 score with 500 neurons and 50 influential neurons is: ", np.around(r2_TN500_IN_50, 2))
print("In general, the more influential neurons the better the R2 score")


# Generate and fit some synthetic spike data as a product of real predictor data + noise

In [ ]:
# Here we link make the design matrix a product of the predictor so we can test the model

def create_synthetic_design_matrix_and_dependent_vector(data):
    """Make the design matrix a product of the predictor so we can test the model"""
    
    # initialise the design matrix and dependent variable
    total_frames = sum([len(homing["frames"]) for homing in data.values()])
    total_neurons = frame_by_cluster_matrix.shape[1] # Total number of neurons
    design_matrix = np.zeros((total_frames, total_neurons))
    dependent_var = np.zeros((total_frames)).T
    assert design_matrix.shape[0] == dependent_var.shape[0], "The design matrix and depedent vector are not the same length"
    
    # print(data)
    counter = 0
    for homing in data.values():
        
        # Extract the frames and predictors
        frames = homing["frames"].to_numpy()
        predictor = homing["predictor"].to_numpy()
        
        # Init
        matrix = np.ones((len(frames), total_neurons))
        # add noise to the matrix
        synthetic_matrix = matrix * predictor[:, None] + np.random.normal(0, 4, (len(frames), total_neurons))
        # Make the design matrix a product of the predictor
        
        # Assign
        design_matrix[0+counter:counter+len(frames)] = synthetic_matrix
        dependent_var[0+counter:counter+len(frames)] = predictor.T
        counter += len(frames)
    
    return design_matrix, dependent_var

synthetic_test_design_matrix, synthetic_test_dependent_var = create_synthetic_design_matrix_and_dependent_vector(test)
synthetic_train_design_matrix, synthetic_train_dependent_var = create_synthetic_design_matrix_and_dependent_vector(train)
# Print the shapes
print(test_design_matrix.shape, test_dependent_var.shape)
print(train_design_matrix.shape, train_dependent_var.shape)

# Fit the model
reg = LinearRegression().fit(synthetic_train_design_matrix, synthetic_train_dependent_var)
r2 = reg.score(synthetic_train_design_matrix, synthetic_train_dependent_var)
print("The synthetic train R2 score is: ", np.around(r2, 2))
print("The synthetic test R2 score is: ", np.around(reg.score(synthetic_test_design_matrix, synthetic_test_dependent_var), 2))


# ----------------------------------------------------- Real tests ---------------------------------------------------------------------------------

In [ ]:
def create_design_matrix_and_dependent_vector(data):
    
    # initialise the design matrix and dependent variable
    # print(data.values())
    total_frames = sum([len(homing["frames"]) for homing in data.values()])
    total_neurons = frame_by_cluster_matrix.shape[1] # Total number of neurons
    design_matrix = np.zeros((total_frames, total_neurons))
    dependent_var = np.zeros((total_frames)).T
    assert design_matrix.shape[0] == dependent_var.shape[0], "The design matrix and depedent vector are not the same length"
    
    # print(data)
    counter = 0
    for homing in data.values():
        
        # Extract the frames and predictors
        frames = homing["frames"].to_numpy()
        predictor = homing["predictor"].to_numpy()
        
        # Assign
        design_matrix[0+counter:counter+len(frames)] = frame_by_cluster_matrix[frames[0]:frames[-1]+1]
        dependent_var[0+counter:counter+len(frames)] = predictor.T
        counter += len(frames)
    
    return design_matrix, dependent_var

# import zscore from scipy
from scipy.stats import zscore


test_design_matrix, test_dependent_var = create_design_matrix_and_dependent_vector(test)
train_design_matrix, train_dependent_var = create_design_matrix_and_dependent_vector(train)

# zscore the design matrix across the columns
train_design_matrix = zscore(train_design_matrix, axis=0)
test_design_matrix = zscore(test_design_matrix, axis=0)

# Replace any NaN values with 0
train_design_matrix = np.nan_to_num(train_design_matrix)
test_design_matrix = np.nan_to_num(test_design_matrix)

# Print the shapes
print(test_design_matrix.shape, test_dependent_var.shape)
print(train_design_matrix.shape, train_dependent_var.shape)

# Fit the model
reg = LinearRegression().fit(train_design_matrix, train_dependent_var)
r2 = reg.score(train_design_matrix, train_dependent_var)
print("The train R2 score on the real data is: ", np.around(r2, 2))
print("The test R2 score on the real data is: ", np.around(reg.score(test_design_matrix, test_dependent_var), 2))

# Plot the residuals vs the predicted values of the model
predicted_values = reg.predict(train_design_matrix)
residuals = train_dependent_var - predicted_values
plt.scatter(predicted_values, residuals)
plt.xlabel("Predicted values")
plt.hlines(y=0, xmin=-1.5, xmax=1, color="red")
plt.ylabel("Residuals")
plt.title("Residuals vs predicted values on train data")
plt.show()

# Plot the residuals vs the predicted values of the model on the test set
predicted_values = reg.predict(test_design_matrix)
residuals = test_dependent_var - predicted_values
plt.scatter(predicted_values, residuals)
plt.xlabel("Predicted values")
plt.hlines(y=0, xmin=-1.5, xmax=1, color="red")
plt.ylabel("Residuals")
plt.title("Residuals vs predicted values")
plt.show()

# Test fit beta regresion

In [ ]:
# import statsmodels.api as sm
# from statsmodels.formula.api import glm
# from statsmodels.genmod.families import Binomial
# from statsmodels.genmod.families.links import logit

# # Transforming the dependent variable
# train_dependent_var_transformed = (train_dependent_var + 1) / 2
# test_dependent_var_transformed = (test_dependent_var + 1) / 2

# # Convert the train and test design matrix to a pandas dataframe
# train_design_matrix = pd.DataFrame(train_design_matrix)
# test_design_matrix = pd.DataFrame(test_design_matrix)

# # Fit Beta Regression
# model_beta = glm(formula='train_dependent_var_transformed ~ train_design_matrix',
#                  family=Binomial(link=logit()),
#                  data=train_design_matrix).fit()

# print("Summary of Beta Regression Model:")
# print(model_beta.summary())

# # Evaluate model
# predictions_train = model_beta.predict(train_design_matrix)

# # Predict on the test set
# predictions_test = model_beta.predict(test_design_matrix)

# # Rescale predictions back to original scale [-1, 1]
# predictions_train_rescaled = predictions_train * 2 - 1
# predictions_test_rescaled = predictions_test * 2 - 1

# train_r2 = r2_score(train_dependent_var, predictions_train_rescaled)


# Test SVR

In [ ]:
from sklearn.svm import SVR

# Fit SVR
svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)  # You can tune these parameters
svr_model.fit(train_design_matrix, train_dependent_var)

# Predictions
predictions_train_svr = svr_model.predict(train_design_matrix)
predictions_test_svr = svr_model.predict(test_design_matrix)

# Print R^2 score
print("Train R2 score from SVR: ", svr_model.score(train_design_matrix, train_dependent_var))
print("Test R2 score from SVR: ", svr_model.score(test_design_matrix, test_dependent_var))

# Plot residuals for SVR
residuals_train_svr = train_dependent_var - predictions_train_svr
plt.scatter(predictions_train_svr, residuals_train_svr)
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("SVR Residuals vs Predicted Values on Train Data")
plt.show()


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

def create_design_matrix_and_dependent_vector(data, scaler=None):
    # Assuming frame_by_cluster_matrix is defined globally
    total_frames = sum(len(homing["frames"]) for homing in data.values())
    total_neurons = frame_by_cluster_matrix.shape[1]  # Total number of neurons
    design_matrix = np.zeros((total_frames, total_neurons))
    dependent_var = np.zeros(total_frames)
    
    counter = 0
    for homing in data.values():
        frames = homing["frames"].to_numpy()
        predictor = homing["predictor"].to_numpy()
        
        design_matrix[counter:counter + len(frames)] = frame_by_cluster_matrix[frames[0]:frames[-1] + 1]
        dependent_var[counter:counter + len(frames)] = predictor
        counter += len(frames)
    
    if scaler is None:
        scaler = StandardScaler()
        design_matrix = scaler.fit_transform(design_matrix)
    else:
        design_matrix = scaler.transform(design_matrix)
    
    return design_matrix, dependent_var, scaler

# Prepare data
# Initialize scaler as None for the train dataset to fit and transform
train_design_matrix, train_dependent_var, scaler = create_design_matrix_and_dependent_vector(train, None)
# Use the fitted scaler for the test dataset
test_design_matrix, test_dependent_var, _ = create_design_matrix_and_dependent_vector(test, scaler)

# Fit the model with regularization
reg = Ridge(alpha=0.1).fit(train_design_matrix, train_dependent_var)
print(f"Train R2 score: {reg.score(train_design_matrix, train_dependent_var):.2f}")
print(f"Test R2 score: {reg.score(test_design_matrix, test_dependent_var):.2f}")

# Plot residuals for training data
predicted_values = reg.predict(train_design_matrix)
residuals = train_dependent_var - predicted_values
plt.scatter(predicted_values, residuals)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values on Train Data")
plt.show()

# Plot residuals for testing data
predicted_values = reg.predict(test_design_matrix)
residuals = test_dependent_var - predicted_values
plt.scatter(predicted_values, residuals)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values on Test Data")
plt.show()


# Test things to handle this weird correlation in the regression

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso


# Generate polynomial features
poly = PolynomialFeatures(degree=1)  # You can adjust the degree
train_design_matrix_poly = poly.fit_transform(train_design_matrix)

# Fit the model on the polynomial features
reg_poly = Lasso(alpha=0.01).fit(train_design_matrix_poly, train_dependent_var)
predicted_values_poly = reg_poly.predict(train_design_matrix_poly)
# print the R2 score
r2_poly = reg_poly.score(train_design_matrix_poly, train_dependent_var)
print("The R2 score on the polynomial features is: ", np.around(r2_poly, 2))

# Fit the poly model on the test data
test_design_matrix_poly = poly.transform(test_design_matrix)
test_predicted_values_poly = reg_poly.predict(test_design_matrix_poly)
# print the R2 score
r2_poly = reg_poly.score(test_design_matrix_poly, test_dependent_var)
print("The R2 score on the polynomial features on the test data is: ", np.around(r2_poly, 2))

# Calculate new residuals
residuals_poly = train_dependent_var - predicted_values_poly

# Plot the new residuals
plt.scatter(predicted_values_poly, residuals_poly)
plt.hlines(y=0, xmin=min(predicted_values_poly), xmax=max(predicted_values_poly), color="red")
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residuals vs predicted values on train data (Polynomial Features)")
plt.show()

## First overfit

In [ ]:

# # reg = LinearRegression().fit(design_matrix, predictor_vector)
# # r2 = reg.score(design_matrix, predictor_vector)
# # print("The overfitted R2 score with all homings is: ", np.around(r2, 2))

# # OLS
# # apply PCA to the design matrix
# from sklearn.decomposition import PCA
# pca = PCA(n_components=rank)
# transform = pca.fit_transform(design_matrix)

# import statsmodels.api as sm
# X = sm.add_constant(transform)
# model = sm.OLS(predictor_vector, X)
# results = model.fit()
# print(results.summary())

## Try splitting data in frames not homings


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_train, X_test, y_train, y_test = train_test_split(design_matrix, predictor_vector, test_size=0.3, random_state=42)
reg = LinearRegression().fit(X_train, y_train)
train_r2 = reg.score(X_train, y_train)
print("The training R2 score: ", np.around(train_r2, 2))
test_r2 = reg.score(X_test, y_test)
print(r"The test R2 score when using 70% of the frames for train is: ", np.around(test_r2, 2))

# OLS ----------------------------------------------------------------------
# apply PCA to the design matrix
from sklearn.decomposition import PCA
pca = PCA(n_components=rank)
transform = pca.fit_transform(design_matrix)
X_train, X_test, y_train, y_test = train_test_split(design_matrix, predictor_vector, test_size=0.3, random_state=42)

import statsmodels.api as sm
X = sm.add_constant(X_train)
model = sm.OLS(y_train, X)
results = model.fit()

# predict on the test set
X = sm.add_constant(X_test)
y_pred = results.predict(X)
r2 = r2_score(y_test, y_pred)
print(f'R² Score: {r2}')




# Try splitting by homing

In [ ]:
reg_homing_split = LinearRegression().fit(train_design_matrix, train_dependent_var)
train_r2 = reg_homing_split.score(train_design_matrix, train_dependent_var)
print("The train R2 score is: ", np.around(train_r2, 2))

# now computer r2 on the test set
test_r2 = reg_homing_split.score(test_design_matrix, test_dependent_var)
print("The test  R2 score is: ", np.around(test_r2, 2))

# from sklearn.decomposition import PCA
# pca = PCA(n_components=50)
# transform = pca.fit_transform(train_design_matrix)

import statsmodels.api as sm
X = sm.add_constant(train_design_matrix)
model = sm.OLS(train_dependent_var, X)
results = model.fit()
predictions = results.predict()
residuals = results.resid
results = model.fit()
# print(results.summary())
# Plot residuals, i.e predicted - actual
# plt.figure(figsize=(10, 6))
# plt.scatter(predictions, residuals, alpha=0.5)
# plt.axhline(y=0, color='r', linestyle='--')
# plt.title('Residual Plot')
# plt.xlabel('Predicted Values')
# plt.ylabel('Residuals (Errors)')
# plt.show()

# Run on test -------------------------------------------------
# pca = PCA(n_components=50)
# transform = pca.fit_transform(test_design_matrix)
X = sm.add_constant(test_design_matrix)
y_pred = results.predict(X)
r2 = r2_score(test_dependent_var, y_pred)
print(f'R² Score for test on OLS model: {r2}')

# cross_val_r2 = reg_homing_split.score(test_design_matrix, test_dependent_var)
# print(test_design_matrix.shape)
# print("The test  R2 score is: ", cross_val_r2)

In [ ]:
# import matplotlib.pyplot as plt

# residuals = interpolated_pred - predictions
# plt.figure(figsize=(10, 6))
# plt.scatter(predictions, residuals)
# plt.axhline(y=0, color='r', linestyle='--')
# plt.xlabel('Predicted Values')
# plt.ylabel('Residuals')
# plt.title('Residual Plot')
# plt.show()


In [ ]:
# plt.figure(figsize=(10, 6))
# plt.scatter(interpolated_pred, predictions, alpha=0.5)
# plt.plot(interpolated_pred, interpolated_pred, color='red')  # Line for perfect prediction
# plt.xlabel('Actual Values')
# plt.ylabel('Predicted Values')
# plt.title('Actual vs. Predicted')
# plt.show()